# 06 · 评测与反馈

评测策略、诊断失败、回流恢复数据，并验证模型更新效果。

In [ ]:
import json
import uuid

from examples._display import (
    setup_notebook,
    show_animation,
    show_image,
    show_metrics,
)

PROJECT_ROOT, OUTPUT_DIR = setup_notebook("evaluation_and_feedback")

from physrealx import ArtifactRef, PhysRealX, RunSpec, VersionedRef
from physrealx.domain import TrainingSpec, canonical_record_id
from physrealx.integrations.act import ACTTrainer
from physrealx.integrations.datasets import dataset_bundle
from physrealx.integrations.feedback import (
    DeploymentEvidence,
    FailureQuery,
    PolicyInterventionEvidence,
)
from physrealx.runtime import load_episode

px = PhysRealX.open(PROJECT_ROOT / ".physrealx")

## 1. 输入

读取已训练策略、评测任务和数据快照。平台在独立闭环 Episode 中评测并诊断失败，生成定向恢复示范，更新策略后使用保留种子进行闭环验证。


In [ ]:
upstream = json.loads(
    (PROJECT_ROOT / "examples/data_management_and_training/output/results.json").read_text()
)
model_ref = VersionedRef.model_validate(upstream["model"])
evaluation_run = RunSpec.model_validate(upstream["evaluation_run"])
snapshot_ref = ArtifactRef.model_validate(upstream["snapshot"])
run_key = model_ref.artifact_id.removeprefix("art_")[:12]

# Collection conditions and reserved evaluation conditions are disjoint.
from examples.web_inputs import module_input
web_inputs = module_input("evaluation_and_feedback")
collection_count = int(web_inputs["collection_count"])
heldout_count = int(web_inputs["heldout_count"])
heldout_seed = int(web_inputs["heldout_seed"])


## 2. 基线评测与日志归档

In [ ]:
evaluation_ref = px.evaluations.submit(
    resource_id=f"laptop1.act.feedback.{run_key}",
    version="2.0.0",
    model=model_ref,
    run=evaluation_run,
    episode_count=3,
    base_seed=101000,
)
evaluation = px.evaluations.get_report(evaluation_ref)
evidence = px.artifacts.put_json(DeploymentEvidence(dataset=evaluation.dataset))
archive = px.telemetry.record(evidence, method="episode_telemetry")
telemetry_ref = archive.output("telemetry")
telemetry_manifest_ref = archive.output("telemetry_manifest")
telemetry_manifest = json.loads(px.artifacts.get_bytes(telemetry_manifest_ref))
show_metrics(
    {
        "基线成功率": evaluation.success_rate,
        "Episode": int(evaluation.metrics["episode_count"]),
        "数据来源": next(iter(telemetry_manifest["source_counts"])),
        "真机证据": telemetry_manifest["live_robot_execution"],
    }
)

## 3. 失败诊断与校准决策

In [ ]:
diagnosis = px.failures.analyze(telemetry_ref, method="trajectory_diagnostics")
failure_report_ref = diagnosis.output("failure_report")
failure_report = json.loads(px.artifacts.get_bytes(failure_report_ref))
calibration_plan_ref = px.feedback.plan_physics_calibration(telemetry_ref, failure_report_ref)
calibration_plan = json.loads(px.artifacts.get_bytes(calibration_plan_ref))
show_metrics(
    {
        "失败 Episode": len(failure_report["failures"]),
        "症状类别": ", ".join(sorted({row["category"] for row in failure_report["failures"]})),
        "物理校准": calibration_plan["status"],
    }
)

## 4. 失败回流与定向恢复

In [ ]:
if not failure_report["failures"]:
    raise RuntimeError("基线评测没有失败样本，无法演示反馈闭环")
failure_key = canonical_record_id({
    "dataset": failure_report["dataset"],
    "episodes": [row["episode_id"] for row in failure_report["failures"]],
})[4:16]
query = px.artifacts.put_json(
    FailureQuery(
        report=failure_report_ref,
        resource_id=f"laptop1.feedback.{failure_key}",
    )
)
selection = px.feedback.build_dataset(query, method="failure_episodes")
selected_dataset = dataset_bundle(px.artifacts, selection.output("dataset"))
selected_ref = px.datasets.register(selected_dataset)
selected = selected_dataset.episodes

recovery_refs, lineage = [], []
for summary in selected:
    failed = load_episode(px.artifacts, summary)
    recovery_key = canonical_record_id({
        "failed_episode": summary.artifact.model_dump(mode="json"),
        "expert": evaluation_run.policy.model_dump(mode="json"),
        "record_signals": evaluation_run.record_signals,
    })[4:16]
    recovery_id = f"laptop1.recovery.{recovery_key}"
    recovery_run = RunSpec(
        run_id=str(
            uuid.uuid5(
                uuid.NAMESPACE_URL,
                f"physrealx:feedback-recovery:{recovery_key}",
            )
        ),
        scene=failed.scene,
        task=failed.task,
        backend=failed.backend,
        policy=evaluation_run.policy,
        episode_count=1,
        base_seed=failed.seed,
        stop_on_success=False,
        dataset_id=recovery_id,
        dataset_version="1.0.0",
        record_signals=evaluation_run.record_signals,
        tags=("expert_demonstration", "failure_recovery", failed.episode_id),
    )
    result = px.runs.submit(recovery_run)
    if result.result is None or result.result.dataset is None:
        raise RuntimeError("定向恢复未产生数据集")
    recovered_ref = result.result.dataset
    recovered = px.datasets.get(recovered_ref)
    if not all(row.success for row in recovered.episodes):
        raise RuntimeError("专家恢复未成功")
    recovery_refs.append(recovered_ref)
    lineage.append(
        {
            "failed_episode": failed.episode_id,
            "recovery_dataset": recovered_ref.model_dump(mode="json"),
            "seed": failed.seed,
        }
    )

intervention_ref = px.artifacts.put_json(
    PolicyInterventionEvidence(
        failure_report=failure_report_ref,
        recovery_datasets=tuple(recovery_refs),
    )
)
causal_report_ref = px.feedback.analyze_policy_intervention(intervention_ref)
causal_report = json.loads(px.artifacts.get_bytes(causal_report_ref))
show_metrics(
    {
        "失败样本": len(selected),
        "恢复示范": len(recovery_refs),
        "配对因果验证": causal_report["validated_count"],
    }
)

## 5. 补充示范


In [ ]:
from physrealx.integrations.mujoco import MujocoConfig

backend_config = MujocoConfig.model_validate(evaluation_run.backend.config)
collection_config = backend_config.model_dump(mode="json")
collection_config["randomization"]["initial_position_jitter"] = 0.02
collection_backend = evaluation_run.backend.model_copy(update={"config": collection_config})
collected_refs = {}
for role, backend, count, seed in (
    ("augmentation", collection_backend, collection_count, 121000),
    ("quality_reference", evaluation_run.backend, heldout_count, heldout_seed),
):
    collection_key = canonical_record_id({
        "scene": evaluation_run.scene.model_dump(mode="json"),
        "task": evaluation_run.task.model_dump(mode="json"),
        "backend": backend.model_dump(mode="json"),
        "expert": evaluation_run.policy.model_dump(mode="json"),
        "record_signals": evaluation_run.record_signals,
        "count": count, "seed": seed, "role": role,
    })[4:16]
    record = px.runs.submit(RunSpec(
        run_id=str(uuid.uuid5(uuid.NAMESPACE_URL, f"physrealx:expert:{collection_key}")),
        scene=evaluation_run.scene,
        task=evaluation_run.task,
        backend=backend,
        policy=evaluation_run.policy,
        episode_count=count,
        base_seed=seed,
        stop_on_success=False,
        dataset_id=f"laptop1.expert.{role}.{collection_key}",
        dataset_version="1.0.0",
        record_signals=evaluation_run.record_signals,
        tags=("expert_demonstration", role),
    ))
    if record.result is None or record.result.dataset is None:
        raise RuntimeError("专家采集没有产生数据集")
    collected_refs[role] = record.result.dataset

attempts = px.datasets.get(collected_refs["augmentation"])
accepted = tuple(row for row in attempts.episodes if row.success)
if not accepted:
    raise RuntimeError("补充采集没有成功示范，请检查专家控制与初始条件")
demonstrations_ref = px.datasets.register(attempts.model_copy(update={
    "resource_id": attempts.resource_id + ".successful",
    "episodes": accepted,
    "source_runs": tuple(dict.fromkeys(row.run_id for row in accepted)),
    "total_steps": sum(row.step_count for row in accepted),
    "metadata": {"collection": collected_refs["augmentation"].model_dump(mode="json"),
                 "selection": "successful_demonstrations"},
}))
reference_dataset_ref = collected_refs["quality_reference"]
demonstration_preview = px.feedback.preview_demonstrations(
    (*recovery_refs, collected_refs["augmentation"])
)
show_image(px.artifacts.get_bytes(demonstration_preview))
show_metrics({"补充采集": len(attempts.episodes), "成功示范": len(accepted),
              "保留失败": len(attempts.episodes) - len(accepted),
              "隔离验收样本": heldout_count})


## 6. 策略更新

In [ ]:
selection_key = canonical_record_id({
    "snapshot": snapshot_ref.model_dump(mode="json"),
    "recoveries": [ref.model_dump(mode="json") for ref in recovery_refs],
    "demonstrations": demonstrations_ref.model_dump(mode="json"),
    "excluded": reference_dataset_ref.model_dump(mode="json"),
})[4:16]
augmented_ref = px.datasets.training_subset(
    snapshot_ref,
    resource_id=f"laptop1.feedback.train.{selection_key}",
    augmentations=(*recovery_refs, demonstrations_ref),
    excluded_datasets=(reference_dataset_ref,),
    required_tag="expert_demonstration",
)
initial_model = px.training.get_model(model_ref)
config = dict(initial_model.training_config["configuration"])
config.update(
    steps=max(600, int(config["steps"])),
    learning_rate=min(1e-4, float(config["learning_rate"])),
)
trainer = ACTTrainer.descriptor.bind(
    checkpoint=ArtifactRef(
        artifact_id=model_ref.artifact_id,
        size_bytes=len(px.artifacts.get_bytes(model_ref.artifact_id)),
        media_type="application/vnd.physrealx.model+json",
    ),
    config=config,
)
training_key = canonical_record_id(
    {"dataset": augmented_ref.model_dump(mode="json"), "trainer": trainer.model_dump(mode="json")}
)[4:16]
updated_model_ref = px.training.submit(
    TrainingSpec(
        resource_id=f"laptop1.act.updated.{training_key}",
        version="1.0.0",
        dataset=augmented_ref,
        trainer=trainer,
    )
)
updated_model = px.training.get_model(updated_model_ref)
if updated_model.weights == initial_model.weights:
    raise RuntimeError("再训练没有产生新权重")
show_metrics({"更新模型": updated_model_ref.resource_id, **updated_model.metrics})

## 7. 独立对照评测

In [ ]:
reports, references = {}, {}
for name, candidate in (("initial", model_ref), ("updated", updated_model_ref)):
    reference = px.evaluations.submit(
        resource_id=f"laptop1.act.heldout.{candidate.artifact_id[4:16]}.{heldout_seed}.{heldout_count}",
        version="2.0.0",
        model=candidate,
        run=evaluation_run,
        episode_count=heldout_count,
        base_seed=heldout_seed,
    )
    report = px.evaluations.get_report(reference)
    reports[name], references[name] = report, reference
policy_update_ref = px.feedback.assess_policy_update(
    references["initial"], references["updated"], reference_dataset=reference_dataset_ref
)
policy_update = json.loads(px.artifacts.get_bytes(policy_update_ref))
show_metrics(
    {
        "初始成功率": reports["initial"].success_rate,
        "更新成功率": reports["updated"].success_rate,
        "策略门控": policy_update["status"],
        "验收依据": policy_update["reason"],
    }
)

## 8. 反馈闭环可视化

In [ ]:
failed_dataset = px.datasets.get(evaluation.dataset)
failed_id = failure_report["failures"][0]["episode_id"]
failed_summary = next(row for row in failed_dataset.episodes if row.episode_id == failed_id)
failed_episode = load_episode(px.artifacts, failed_summary)
recovery_dataset = px.datasets.get(recovery_refs[0])
recovery_summary = recovery_dataset.episodes[0]

replay_ref = px.evaluations.submit(
    resource_id=f"laptop1.act.replay.updated.{training_key}",
    version="2.0.0",
    model=updated_model_ref,
    run=evaluation_run,
    episode_count=1,
    base_seed=failed_episode.seed,
)
replay_report = px.evaluations.get_report(replay_ref)
replay_summary = px.datasets.get(replay_report.dataset).episodes[0]

dashboard_ref = px.feedback.preview_report(
    references["initial"],
    references["updated"],
    failure_report_ref,
    causal_report_ref,
    calibration_plan_ref,
    policy_update_ref,
)
cycle_ref = px.feedback.preview_cycle(
    failed_summary.artifact,
    recovery_summary.artifact,
    replay_summary.artifact,
    frames=8,
    fps=3,
)
show_image(px.artifacts.get_bytes(dashboard_ref))
show_animation(px.artifacts.get_bytes(cycle_ref))

## 9. 输出

In [ ]:
lineage_ref = px.artifacts.put_json(
    {
        "schema": "physrealx.recovery-lineage/3",
        "links": lineage,
        "initial_model": model_ref.model_dump(mode="json"),
        "updated_model": updated_model_ref.model_dump(mode="json"),
        "training_dataset": augmented_ref.model_dump(mode="json"),
        "paired_success_rates": {key: value.success_rate for key, value in reports.items()},
        "policy_update_status": policy_update["status"],
    }
)
results = {
    "initial_model": model_ref.model_dump(mode="json"),
    "model": updated_model_ref.model_dump(mode="json"),
    "evaluations": {name: ref.model_dump(mode="json") for name, ref in references.items()},
    "failure_report": failure_report_ref.model_dump(mode="json"),
    "causal_failure_report": causal_report_ref.model_dump(mode="json"),
    "physics_calibration_plan": calibration_plan_ref.model_dump(mode="json"),
    "policy_update_assessment": policy_update_ref.model_dump(mode="json"),
    "feedback_dataset": selected_ref.model_dump(mode="json"),
    "training_dataset": augmented_ref.model_dump(mode="json"),
    "recovery_datasets": [ref.model_dump(mode="json") for ref in recovery_refs],
    "demonstration_attempts": collected_refs["augmentation"].model_dump(mode="json"),
    "demonstrations": demonstrations_ref.model_dump(mode="json"),
    "quality_reference": reference_dataset_ref.model_dump(mode="json"),
    "demonstration_preview": demonstration_preview.model_dump(mode="json"),
    "feedback_dashboard": dashboard_ref.model_dump(mode="json"),
    "feedback_cycle": cycle_ref.model_dump(mode="json"),
}
px.artifacts.export(
    OUTPUT_DIR,
    results,
    files={
        "failure-report.json": failure_report_ref,
        "causal-failure-report.json": causal_report_ref,
        "physics-calibration-plan.json": calibration_plan_ref,
        "policy-update-assessment.json": policy_update_ref,
        "recovery-lineage.json": lineage_ref,
        "feedback-dashboard.png": dashboard_ref,
        "demonstrations.png": demonstration_preview,
        "feedback-cycle.gif": cycle_ref,
    },
)
print("结果索引:", OUTPUT_DIR / "results.json")